# NERVE Dataset Generation Pipeline

NERVE turns a **raw session folder** (RGB, DVS, radar, COCO labels, timings)
into **training-ready datasets** for three model families:

| Model family | Template prefix | Event repr | Clip mode | Storage |
|---|---|---|---|---|
| **YOLOX / YOLOv8** | `yolox_yolov8_*` | histogram PNG or VTEI PNG | `single_frame` | PNG + COCO JSON |
| **ReYOLOv8** | `reyolov8_*` | VTEI (raw events) | `sequence` | HDF5 sequences + `.npy` labels |
| **RVT** | `rvt_*` | stacked histogram | `sequence` | per-sequence dirs with `event_representations.h5` + `labels.npz` |

Configuration is JSON-based: each template specifies sensor streams, resolutions,
event representations, optional radar fusion, and class filters.


In [4]:
from nerve import config, remote
from nerve.registry import all_sessions
from pathlib import Path

sessions = all_sessions()
smallest = min(sessions, key=lambda s: s.size_bytes)
print(f"Smallest session: {smallest.name}  "
      f"({smallest.size_bytes / 1e9:.2f} GB archive, split={smallest.split})")

session_dir = Path(remote.download_session(smallest.name))
print(f"Session directory: {session_dir}")


Smallest session: 2023-10-26_15-37-59  (0.40 GB archive, split=train)
  Session already exists: /gpfs/home5/omansour/.nerve/data/2023-10-26_15-37-59
Session directory: /gpfs/home5/omansour/.nerve/data/2023-10-26_15-37-59


---
## Available templates

Templates are bundled under `nerve.generation.templates`. Each is a JSON array of
data-source blocks specifying the sensor, event representation, clip mode, and output
format. The table below lists all shipped templates.


In [5]:
import json
from importlib.resources import files as _pkg_files

tmpl_dir = _pkg_files("nerve.generation.templates")
header = f"{'Template':<46s} {'Format':<13s} {'Clip mode':<16s} {'HDF5':<6s} {'Event repr':<16s} Sources"
print(header)
print("-" * len(header))
for p in sorted(tmpl_dir.iterdir()):
    if not p.name.endswith(".template.json"):
        continue
    s = json.loads(p.read_text(encoding="utf-8"))
    blocks = [b for b in s if isinstance(b, dict)]
    first = blocks[0] if blocks else {}
    srcs = [b["data"] for b in blocks if "data" in b]
    print(f"{p.name:<46s} {first.get('output_format','yolox_png'):<13s} "
          f"{first.get('clip_mode','?'):<16s} {str(first.get('store_as_hdf5',False)):<6s} "
          f"{first.get('event_representation','histogram'):<16s} {srcs}")


Template                                       Format        Clip mode        HDF5   Event repr       Sources
-------------------------------------------------------------------------------------------------------------
reyolov8_distance.template.json                yolox_png     sequence         True   vtei             ['davis', 'ti_radar']
reyolov8_distance_prop.template.json           yolox_png     sequence         True   vtei             ['prophesee', 'ti_radar']
reyolov8_sequence.template.json                yolox_png     sequence         True   vtei             ['davis']
rvt_distance.template.json                     rvt           sequence         False  stacked_histogram ['davis', 'ti_radar']
rvt_distance_prop.template.json                rvt           sequence         False  stacked_histogram ['prophesee', 'ti_radar']
rvt_sequence.template.json                     rvt           sequence         False  stacked_histogram ['davis']
yolox_yolov8_distance.template.json            yo

---
## Comparing YOLOX, ReYOLOv8, and RVT templates

Below we inspect one template from each model family to highlight the key
differences in configuration.


In [6]:
templates = [
    ("YOLOX / YOLOv8 (PNG + distance)", "yolox_yolov8_distance.template.json"),
    ("ReYOLOv8 (HDF5 sequence + distance)", "reyolov8_distance.template.json"),
    ("RVT (sequence + distance)", "rvt_distance.template.json"),
]

for label, fname in templates:
    p = _pkg_files("nerve.generation.templates").joinpath(fname)
    settings = json.loads(p.read_text(encoding="utf-8"))
    blocks = [b for b in settings if isinstance(b, dict)]
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    for block in blocks:
        if "data" in block:
            print(f"\n  --- source: {block['data']} ---")
            for k in ["output_format", "clip_mode", "store_as_hdf5",
                      "use_raw_events", "event_representation", "bins",
                      "sequence_length", "clip_length", "clip_stride",
                      "fuse_dvs_radar_png", "fuse_radar_channel",
                      "output_shape", "mapping"]:
                if k in block:
                    print(f"    {k}: {block[k]}")



  YOLOX / YOLOv8 (PNG + distance)

  --- source: davis ---
    clip_mode: single_frame
    store_as_hdf5: False
    use_raw_events: False
    fuse_dvs_radar_png: True
    output_shape: [346, 260]
    mapping: $NERVE_MAPPINGS/rgb_to_davis.json

  --- source: ti_radar ---
    output_shape: [346, 260]
    mapping: $NERVE_MAPPINGS/ti_radar_to_davis.json

  ReYOLOv8 (HDF5 sequence + distance)

  --- source: davis ---
    clip_mode: sequence
    store_as_hdf5: True
    use_raw_events: True
    event_representation: vtei
    bins: 5
    clip_length: 11
    clip_stride: 11
    output_shape: [384, 288]
    mapping: $NERVE_MAPPINGS/rgb_to_davis.json

  --- source: ti_radar ---
    output_shape: [384, 288]
    mapping: $NERVE_MAPPINGS/ti_radar_to_davis.json

  RVT (sequence + distance)

  --- source: davis ---
    output_format: rvt
    clip_mode: sequence
    use_raw_events: True
    event_representation: stacked_histogram
    bins: 5
    sequence_length: 11
    output_shape: [384, 320]
    map

---
## Pipeline steps

End-to-end flow implemented in `nerve.generation.creator.extract_from_single_session`:

1. **Load timing offsets** from `timings.json` and align all sensor streams.
2. **Read events / frames** from the selected sensor (DAVIS, Prophesee, or both).
3. **Build representation**: `process_events` for VTEI / stacked histogram / voxel grid,
   or histogram-based PNG rasterisation.
4. **MapLabels** using RGB→DVS (and radar→DVS) extrinsics from the calibration JSONs.
5. **Optional radar fusion**: fuse TI radar distance channel or DVS-radar PNG composite.
6. **Write output** in the chosen format:

| Format | Writer | Output structure |
|--------|--------|------------------|
| **YOLOX PNG** | `store_data` + `LabelWriter` | `{split}/data/{sensor}/` + `{split}/annotations/` (COCO JSON) |
| **ReYOLOv8 HDF5** | `store_sequence_data` | `{split}/data/{sensor}/sequence_*.h5` (key `1mp`) + `{split}/labels/*.npy` |
| **RVT** | `store_rvt_sequence_data` | `{split}/sequence_NNNNNN/event_representations_v2/`, `labels_v2/`, `timestamps_us.npy` |


---
## Output directory structures

### YOLOX / YOLOv8 (PNG)
```
{dest}/{split}/
├── data/{davis|prophesee}/      # event representation PNGs
├── annotations/{sensor}.json    # COCO-format labels
└── data.yaml                    # YOLO training config
```

### ReYOLOv8 (HDF5)
```
{dest}/{split}/
├── data/{davis|prophesee}/
│   └── sequence_00_subseq_XXXXXXX.h5   # dataset key: "1mp"
├── labels/
│   └── sequence_00_subseq_XXXXXXX.npy   # per-frame (N, 6) boxes
└── data.yaml
```

### RVT
```
{dest}/{split}/
├── sequence_000000/
│   ├── event_representations_v2/{ev_repr_name}/
│   │   └── event_representations.h5       # dataset key: "data"
│   ├── labels_v2/
│   │   ├── labels.npz                     # keys: labels, objframe_idx_2_label_idx
│   │   └── timestamps_us.npy
│   ├── timestamps_us.npy
│   └── objframe_idx_2_repr_idx.npy
├── sequence_000001/
│   └── ...
└── data.yaml
```


---
## CLI: `nerve generate`

The generation pipeline is accessible via the `nerve generate` CLI. Key arguments:

| Flag | Description |
|------|-------------|
| `--template` | Template name (e.g. `rvt_distance`) or path to `.json` |
| `--dest` | Output dataset directory |
| `--split` | Filter sessions by split **and** set the output subdirectory |
| `--split-label` | Override the output subdirectory name |
| `--from-file` | Session list file (one name per line) |
| `--clean` | Remove existing output before generating |
| `--add` | Append to an existing dataset |
| `--verbose` | Enable verbose output |


In [7]:
examples = [
    ("YOLOX / YOLOv8 (PNG, single-frame, DAVIS + radar distance)",
     "nerve generate --template yolox_yolov8_distance --dest /tmp/nerve_yolox "
     "--split train --clean --verbose"),
    ("ReYOLOv8 (HDF5 sequences, DAVIS + radar distance)",
     "nerve generate --template reyolov8_distance --dest /tmp/nerve_reyolo "
     "--split train --clean --verbose"),
    ("RVT (stacked histogram sequences, DAVIS + radar distance)",
     "nerve generate --template rvt_distance --dest /tmp/nerve_rvt "
     "--split train --clean --verbose"),
    ("RVT (detection only, DAVIS, no radar)",
     "nerve generate --template rvt_sequence --dest /tmp/nerve_rvt_seq "
     "--split val --clean --verbose"),
]

for label, cmd in examples:
    print(f"# {label}")
    print(f"  {cmd}\n")


# YOLOX / YOLOv8 (PNG, single-frame, DAVIS + radar distance)
  nerve generate --template yolox_yolov8_distance --dest /tmp/nerve_yolox --split train --clean --verbose

# ReYOLOv8 (HDF5 sequences, DAVIS + radar distance)
  nerve generate --template reyolov8_distance --dest /tmp/nerve_reyolo --split train --clean --verbose

# RVT (stacked histogram sequences, DAVIS + radar distance)
  nerve generate --template rvt_distance --dest /tmp/nerve_rvt --split train --clean --verbose

# RVT (detection only, DAVIS, no radar)
  nerve generate --template rvt_sequence --dest /tmp/nerve_rvt_seq --split val --clean --verbose



---
## Train / val / test splits

Each session in the NERVE registry has a fixed `split` field. Use `nerve.registry`
to inspect the distribution and export session lists for the generation CLI.


In [8]:
from collections import Counter

split_counts = Counter(s.split for s in sessions)
total_size = {}
for s in sessions:
    total_size[s.split] = total_size.get(s.split, 0) + s.size_bytes

header = f"{'Split':<8s} {'Sessions':>10s} {'Total size':>12s}"
print(header)
print("-" * len(header))
for split in ["train", "val", "test"]:
    n = split_counts.get(split, 0)
    gb = total_size.get(split, 0) / 1e9
    print(f"{split:<8s} {n:>10d} {gb:>10.1f} GB")
print(f"{'total':<8s} {len(sessions):>10d} {sum(total_size.values())/1e9:>10.1f} GB")


Split      Sessions   Total size
--------------------------------
train            94      383.0 GB
val              10       39.4 GB
test             11       33.4 GB
total           116      462.6 GB
